# 한국 배출권 연평균 가격 구하기

In [1]:
import pandas as pd
import os

# 1. 2018년부터 2026년까지의 파일 이름을 리스트로 생성
years = range(2018, 2027)
file_names = [f"../csv/raw/ets/{year}.csv" for year in years]

df_list = []

# 2. 반복문을 돌며 파일들을 하나씩 읽어와 리스트에 담기
print("데이터를 불러오는 중입니다...")
for file in file_names:
    if os.path.exists(file): # 파일이 같은 폴더에 있는지 확인
        # 쉼표(,)가 포함된 숫자를 문자가 아닌 숫자로 정상 인식하도록 thousands=',' 옵션 유지
        temp_df = pd.read_csv(file, thousands=',')
        df_list.append(temp_df)
    else:
        print(f"🚨 주의: {file} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")

# 3. 리스트에 담긴 여러 연도의 데이터를 하나의 데이터프레임으로 병합
df_total = pd.concat(df_list, ignore_index=True)
print("모든 데이터 병합 완료!")

# 4. 병합된 데이터의 '일자' 칼럼에서 정확한 거래 '연도'만 추출
df_total['연도'] = pd.to_datetime(df_total['일자']).dt.year

# 5. KAU 종목이면서, 유효한 거래(거래량 > 0)가 발생한 날만 필터링
df_kau = df_total[(df_total['종목명'].str.contains('KAU')) & (df_total['거래량'] > 0)]

# 6. 연도별로 그룹화하여 총 거래대금과 총 거래량 합산
annual_data = df_kau.groupby('연도')[['거래대금', '거래량']].sum()

# 7. 연도별 가중평균 가격 산출 = (총 거래대금 / 총 거래량)
annual_data['연평균가격'] = annual_data['거래대금'] / annual_data['거래량']

# 결과를 보기 좋게 정수형으로 반올림 처리 (선택 사항)
annual_data['연평균가격'] = annual_data['연평균가격'].round(0).astype(int)

# 최종 결과 출력
print("\n[연도별 K-ETS(KAU) 가중평균 가격]")
print(annual_data[['연평균가격']])

데이터를 불러오는 중입니다...
모든 데이터 병합 완료!

[연도별 K-ETS(KAU) 가중평균 가격]
      연평균가격
연도         
2018  22237
2019  29126
2020  29026
2021  19709
2022  20633
2023   9987
2024   9245
2025   9170
2026  15229


In [2]:
# 8. 계산된 연평균 가격 데이터를 CSV 파일로 저장
# 한글 깨짐 방지를 위해 encoding='utf-8-sig' 옵션을 반드시 추가합니다.
annual_data[['연평균가격']].to_csv('../csv/processed/KAU_연평균가격.csv', encoding='utf-8-sig')

print("✅ '../csv/processed/KAU_연평균가격.csv' 파일로 저장이 완료되었습니다!")

✅ '../csv/processed/KAU_연평균가격.csv' 파일로 저장이 완료되었습니다!


# 달러, 유로화 기준 가격도 구하기(최종)

In [4]:
!pip install yfinance

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 8.9 MB/s eta 0:00:00
  Attempting uninstall: cffi
    Found existing installation: cffi 1.17.1
    Uninstalling cffi-1.17.1:
      Successfully uninstalled cffi-1.17.1


In [5]:
import pandas as pd
import os
import yfinance as yf

# ==========================================
# 1. K-ETS 배출권 일별 데이터 불러오기 및 병합
# ==========================================
years = range(2018, 2027)
file_names = [f"../csv/raw/ets/{year}.csv" for year in years]
df_list = []

print("K-ETS 데이터 및 일별 환율 데이터를 불러오는 중입니다...")
for file in file_names:
    if os.path.exists(file):
        temp_df = pd.read_csv(file, thousands=',')
        df_list.append(temp_df)
    else:
        print(f"🚨 주의: {file} 파일을 찾을 수 없습니다.")

df_total = pd.concat(df_list, ignore_index=True)

# '일자' 칼럼을 날짜 형식(datetime)으로 변환
df_total['일자'] = pd.to_datetime(df_total['일자'])

# ==========================================
# 2. 날짜 필터링 (2026년 5월 15일까지만 포함)
# ==========================================
target_date = pd.to_datetime('2026-05-15')
df_total = df_total[df_total['일자'] <= target_date]

# 연도 칼럼 추가 및 대상 종목(KAU, 거래량 > 0) 필터링
df_total['연도'] = df_total['일자'].dt.year
df_kau = df_total[(df_total['종목명'].str.contains('KAU')) & (df_total['거래량'] > 0)].copy()

# ==========================================
# 3. 일별 환율 데이터 수집 (야후 파이낸스 API)
# ==========================================
# USD/KRW 환율 수집 (2018-01-01 ~ 2026-05-16, yfinance는 end date 하루 전까지 불러옴)
usd_krw = yf.download('KRW=X', start='2018-01-01', end='2026-05-16')[['Close']]
usd_krw.columns = ['USD_KRW'] # 컬럼명 단순화

# EUR/KRW 환율 수집
eur_krw = yf.download('EURKRW=X', start='2018-01-01', end='2026-05-16')[['Close']]
eur_krw.columns = ['EUR_KRW']

# 두 환율 데이터 병합 및 인덱스(날짜) 초기화
fx_data = pd.merge(usd_krw, eur_krw, left_index=True, right_index=True, how='outer')
fx_data = fx_data.reset_index()
fx_data.rename(columns={'Date': '일자'}, inplace=True)
fx_data['일자'] = pd.to_datetime(fx_data['일자'])

# 주말/공휴일 등 환율 시장 휴장일은 금요일(직전 영업일) 환율로 채우기 (ffill)
fx_data = fx_data.ffill()

# ==========================================
# 4. K-ETS 데이터와 환율 데이터 결합 및 계산
# ==========================================
# 배출권 일자 기준으로 환율 데이터 Left Join
df_kau = pd.merge(df_kau, fx_data, on='일자', how='left')

# 혹시 모를 결측치 대비 한 번 더 직전/직후 영업일 채우기
df_kau['USD_KRW'] = df_kau['USD_KRW'].ffill().bfill()
df_kau['EUR_KRW'] = df_kau['EUR_KRW'].ffill().bfill()

# [핵심 로직] 일별 당일 환율을 적용하여 달러화/유로화 거래대금 환산
df_kau['거래대금_USD'] = df_kau['거래대금'] / df_kau['USD_KRW']
df_kau['거래대금_EUR'] = df_kau['거래대금'] / df_kau['EUR_KRW']

# ==========================================
# 5. 연도별 그룹화 및 최종 다통화 가중평균 산출
# ==========================================
# 연도별로 총 거래량, 원화/달러화/유로화 총 거래대금 합산
annual_data = df_kau.groupby('연도')[['거래량', '거래대금', '거래대금_USD', '거래대금_EUR']].sum()

# 합산된 총 거래대금을 총 거래량으로 나누어 최종 VWAP 산출
annual_data['VWAP_KRW'] = annual_data['거래대금'] / annual_data['거래량']
annual_data['VWAP_USD'] = annual_data['거래대금_USD'] / annual_data['거래량']
annual_data['VWAP_EUR'] = annual_data['거래대금_EUR'] / annual_data['거래량']

# 데이터 깔끔하게 포맷팅 (원화는 정수, 달러/유로는 소수점 둘째 자리)
annual_data['VWAP_KRW'] = annual_data['VWAP_KRW'].round(0).astype(int)
annual_data['VWAP_USD'] = annual_data['VWAP_USD'].round(2)
annual_data['VWAP_EUR'] = annual_data['VWAP_EUR'].round(2)

result_df = annual_data[['VWAP_KRW', 'VWAP_USD', 'VWAP_EUR']]

print("\n[연도별 K-ETS(KAU) 다통화 가중평균 가격 (2026.05.15 기준)]")
print(result_df)

# ==========================================
# 6. CSV 파일로 저장
# ==========================================
result_df.to_csv('../csv/processed/KAU_다통화_연평균가격.csv', encoding='utf-8-sig')
print("\n✅ '../csv/processed/KAU_다통화_연평균가격.csv' 파일로 저장이 완료되었습니다!")

K-ETS 데이터 및 일별 환율 데이터를 불러오는 중입니다...


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


[연도별 K-ETS(KAU) 다통화 가중평균 가격 (2026.05.15 기준)]
      VWAP_KRW  VWAP_USD  VWAP_EUR
연도                                
2018     22237     20.62     17.39
2019     29126     24.74     22.16
2020     29026     24.30     21.53
2021     19709     17.26     14.62
2022     20633     16.18     15.28
2023      9987      7.69      7.06
2024      9245      6.66      6.25
2025      9170      6.45      5.78
2026     15221     10.30      8.82

✅ '../csv/processed/KAU_다통화_연평균가격.csv' 파일로 저장이 완료되었습니다!


# 이번엔 EU 배출권 연평균 가격 구하기

In [6]:
import pandas as pd
import yfinance as yf

# ==========================================
# 1. EU 배출권 데이터 불러오기 및 날짜 처리
# ==========================================
print("EU 배출권 데이터 및 환율 데이터를 불러오는 중입니다...")
df_eu = pd.read_csv('../csv/raw/EU_2018_2026.csv')

# 날짜 칼럼 변환 ('MM/DD/YYYY' 형식을 datetime으로 인식)
df_eu['Date'] = pd.to_datetime(df_eu['Date'])

# 2026년 5월 15일까지만 데이터 필터링 (시점 통일)
target_date = pd.to_datetime('2026-05-15')
df_eu = df_eu[df_eu['Date'] <= target_date].copy()

# 연도 칼럼 추가
df_eu['Year'] = df_eu['Date'].dt.year

# ==========================================
# 2. 거래량(Vol.) 문자열 전처리 및 숫자 변환
# ==========================================
def parse_volume(vol_str):
    if pd.isna(vol_str):
        return 0
    vol_str = str(vol_str).strip().upper()
    if vol_str == '-':
        return 0
    vol_str = vol_str.replace(',', '') # 혹시 모를 쉼표 제거
    
    # K(천), M(백만), B(십억) 단위 환산
    if vol_str.endswith('K'):
        return float(vol_str[:-1]) * 1e3
    elif vol_str.endswith('M'):
        return float(vol_str[:-1]) * 1e6
    elif vol_str.endswith('B'):
        return float(vol_str[:-1]) * 1e9
    else:
        return float(vol_str)

df_eu['Volume'] = df_eu['Vol.'].apply(parse_volume)

# 거래량이 0 이상인 유효 거래만 남기기
df_eu = df_eu[df_eu['Volume'] > 0].copy()

# Price(종가) 컬럼 숫자형 변환
df_eu['Price'] = df_eu['Price'].astype(str).str.replace(',', '').astype(float)

# ==========================================
# 3. 유로화(EUR) 기준 기본 거래대금 산출
# ==========================================
df_eu['거래대금_EUR'] = df_eu['Price'] * df_eu['Volume']

# ==========================================
# 4. 일별 환율 데이터 수집 (야후 파이낸스)
# ==========================================
usd_krw = yf.download('KRW=X', start='2018-01-01', end='2026-05-16')[['Close']]
usd_krw.columns = ['USD_KRW']

eur_krw = yf.download('EURKRW=X', start='2018-01-01', end='2026-05-16')[['Close']]
eur_krw.columns = ['EUR_KRW']

# 환율 데이터 병합 (Outer Join)
fx_data = pd.merge(usd_krw, eur_krw, left_index=True, right_index=True, how='outer').reset_index()
fx_data.rename(columns={'Date': 'Date'}, inplace=True)
fx_data['Date'] = pd.to_datetime(fx_data['Date'])
fx_data = fx_data.ffill() # 주말 등 휴장일 데이터 채우기

# ==========================================
# 5. EUA 데이터와 환율 데이터 결합 및 다통화 계산
# ==========================================
# 배출권 일자 기준으로 환율 Left Join
df_eu = pd.merge(df_eu, fx_data, on='Date', how='left')

# 결측치 방어
df_eu['EUR_KRW'] = df_eu['EUR_KRW'].ffill().bfill()
df_eu['USD_KRW'] = df_eu['USD_KRW'].ffill().bfill()

# [핵심 로직] 유로화 대금을 환율을 이용해 원화/달러화로 환산
df_eu['거래대금_KRW'] = df_eu['거래대금_EUR'] * df_eu['EUR_KRW']
df_eu['거래대금_USD'] = df_eu['거래대금_KRW'] / df_eu['USD_KRW']

# ==========================================
# 6. 연도별 그룹화 및 최종 VWAP 산출
# ==========================================
annual_eu = df_eu.groupby('Year')[['Volume', '거래대금_EUR', '거래대금_KRW', '거래대금_USD']].sum()

# 합산된 총 거래대금을 총 거래량으로 나누기
annual_eu['VWAP_EUR'] = annual_eu['거래대금_EUR'] / annual_eu['Volume']
annual_eu['VWAP_KRW'] = annual_eu['거래대금_KRW'] / annual_eu['Volume']
annual_eu['VWAP_USD'] = annual_eu['거래대금_USD'] / annual_eu['Volume']

# 데이터 포맷팅
annual_eu['VWAP_EUR'] = annual_eu['VWAP_EUR'].round(2)
annual_eu['VWAP_USD'] = annual_eu['VWAP_USD'].round(2)
annual_eu['VWAP_KRW'] = annual_eu['VWAP_KRW'].round(0).astype(int)

result_eu = annual_eu[['VWAP_EUR', 'VWAP_KRW', 'VWAP_USD']]

print("\n[연도별 EUA 다통화 가중평균 가격 (2026.05.15 기준)]")
print(result_eu)

# ==========================================
# 7. CSV 파일로 저장
# ==========================================
result_eu.to_csv('../csv/processed/EUA_다통화_연평균가격.csv', encoding='utf-8-sig')
print("\n✅ '../csv/processed/EUA_다통화_연평균가격.csv' 파일로 저장이 완료되었습니다!")

[*********************100%***********************]  1 of 1 completed

EU 배출권 데이터 및 환율 데이터를 불러오는 중입니다...



[*********************100%***********************]  1 of 1 completed


[연도별 EUA 다통화 가중평균 가격 (2026.05.15 기준)]
      VWAP_EUR  VWAP_KRW  VWAP_USD
Year                              
2018     16.66     21583     19.48
2019     24.69     32169     27.64
2020     24.52     32953     28.07
2021     54.37     73576     63.94
2022     80.64    109273     85.57
2023     84.77    119331     91.46
2024     66.30     97560     71.63
2025     74.88    120046     84.28
2026     76.76    131720     89.96

✅ '../csv/processed/EUA_다통화_연평균가격.csv' 파일로 저장이 완료되었습니다!
